# EQ/FX Hybrid: SPX Call Contingent on USDJPY > Barrier

A walkthrough of pricing and risk analysis for the cross-asset payoff:

$$V_T = \max(S_T - K, 0) \cdot \mathbf{1}\{X_T > B\}$$

where $S$ = SPX, $X$ = USDJPY, $K$ = equity strike, $B$ = FX barrier.

**Sections:**
1. Setup and the example trade
2. Closed-form price and Monte Carlo cross-check
3. Greeks scaled to notional
4. SPX delta — well-behaved but FX-conditional
5. SPX vega — bell-shaped, fanned by FX
6. FX vega — sign flip at the barrier
7. Cega — correlation risk
8. Cross-gamma surface
9. Scenario walkthrough: when delta-hedging fails

## 1. Setup

The closed-form pricer in `src/hybrid_pricer.py` implements:

$$V_0 = S_0 e^{-qT}\, M_2(d_1^S,\; d_2^X + \rho\sigma_S\sqrt{T};\; \rho)\; -\; K e^{-r_d T}\, M_2(d_2^S,\; d_2^X;\; \rho)$$

under the joint lognormal model with correlation $\rho$ between SPX and USDJPY.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace

from src.hybrid_pricer import (
    HybridInputs, price_hybrid_call, price_hybrid_call_mc, greeks
)
from src.trade_config import EXAMPLE_TRADE as p, NOTIONAL

print(f"Notional: ${NOTIONAL/1e6:.0f}mm")
print(f"SPX: spot {p.S0}, strike {p.K}  ({(p.S0/p.K-1)*100:+.1f}% ITM)")
print(f"USDJPY: spot {p.X0}, barrier {p.B}  ({(p.B/p.X0-1)*100:+.1f}% above spot)")
print(f"T={p.T}y, sig_S={p.sig_S:.0%}, sig_X={p.sig_X:.0%}, rho={p.rho}")

## 2. Price: closed form vs Monte Carlo

The closed-form should match Monte Carlo to within standard error — a key sanity check given the measure-change derivation.

In [ ]:
res = price_hybrid_call(p)
mc = price_hybrid_call_mc(p, n_paths=1_000_000, seed=42)

units = NOTIONAL / p.S0

print(f"Closed form:           ${res['price']:.4f} per SPX unit")
print(f"Monte Carlo (1M):      ${mc['price']:.4f}  (95% CI [{mc['ci95'][0]:.4f}, {mc['ci95'][1]:.4f}])")
print()
print(f"Premium $:             ${res['price']*units/1e6:.2f}mm")
print(f"Vanilla equivalent:    ${res['vanilla_SPX_call']*units/1e6:.2f}mm")
print(f"Hybrid / Vanilla:      {res['price']/res['vanilla_SPX_call']:.1%}")
print(f"P(joint exercise):     {res['P_joint_exercise']:.1%}")
print(f"P(USDJPY > B):         {res['P_FX_above_barrier']:.1%}")

**Read:** the buyer pays ~59% of the vanilla premium for the FX-conditional payoff. The trade only pays in ~33% of risk-neutral scenarios.

## 3. Greeks scaled to notional

In [ ]:
g = greeks(p)

print("Greeks scaled to $100mm notional:")
print(f"  Delta SPX:   {g['delta_SPX']:.4f}     ->  ${g['delta_SPX']*NOTIONAL/1e6:.1f}mm SPX equivalent")
print(f"  Delta FX:    {g['delta_FX']:.2f}     ->  ${g['delta_FX']*units/1e3:.1f}k per yen USDJPY")
print(f"  Vega SPX:    {g['vega_SPX']:.2f}     ->  ${g['vega_SPX']*units/1e3:.1f}k per vol pt")
print(f"  Vega FX:     {g['vega_FX']:.2f}     ->  ${g['vega_FX']*units/1e3:.1f}k per vol pt")
print(f"  Cega:        {g['cega_corr']:.3f}     ->  ${g['cega_corr']*units/1e3:.1f}k per 1% corr")

**Cega** is the unhedgeable risk. At ~$30k per 1% correlation, a typical ±10 corr-point uncertainty implies ±$300k of valuation gap — about 7% of premium.

## 4. Helper functions

We re-bind the trade with different spots/inputs throughout.

In [ ]:
def price_at(S0, X0, **overrides):
    return price_hybrid_call(replace(p, S0=S0, X0=X0, **overrides))["price"]

def vega_at(S0, X0, which="sig_S", dv=0.01, **ov):
    up = price_at(S0, X0, **{**ov, which: getattr(p, which) + dv})
    dn = price_at(S0, X0, **{**ov, which: getattr(p, which) - dv})
    return (up - dn) / 2

def cega_at(S0, X0, drho=0.01, **ov):
    up = price_at(S0, X0, **{**ov, "rho": p.rho + drho})
    dn = price_at(S0, X0, **{**ov, "rho": p.rho - drho})
    return (up - dn) / 2

def delta_at(S0, X0, dS=None, **ov):
    dS = dS or 0.005 * p.K
    return (price_at(S0 + dS, X0, **ov) - price_at(S0 - dS, X0, **ov)) / (2 * dS)

## 5. SPX delta — well-behaved, but the *hedge size* depends on FX

Familiar S-curve, uniformly compressed by the FX barrier probability. The right panel shows the same data as a notional-equivalent hedge size.

In [ ]:
spx_grid = np.linspace(6200, 8000, 80)
fx_levels = [150, 153, 156, 159, 162]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for X0 in fx_levels:
    deltas = [delta_at(s, X0) for s in spx_grid]
    ax.plot(spx_grid, deltas, label=f"USDJPY = {X0}", linewidth=2)
vanilla = [delta_at(s, 1000.0) for s in spx_grid]
ax.plot(spx_grid, vanilla, "k--", alpha=0.6, label="Vanilla")
ax.axvline(p.K, color="gray", ls=":", alpha=0.6, label=f"Strike {p.K:.0f}")
ax.axvline(p.S0, color="red", ls=":", alpha=0.5)
ax.set_xlabel("SPX spot"); ax.set_ylabel("SPX delta")
ax.set_title("SPX Delta vs Spot")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for X0 in fx_levels:
    ne = [delta_at(s, X0) * NOTIONAL / 1e6 for s in spx_grid]
    ax.plot(spx_grid, ne, label=f"USDJPY = {X0}", linewidth=2)
ax.axvline(p.K, color="gray", ls=":", alpha=0.6)
ax.axvline(p.S0, color="red", ls=":", alpha=0.5)
ax.set_xlabel("SPX spot"); ax.set_ylabel("Hedge size ($mm)")
ax.set_title(f"Notional-equivalent hedge on ${NOTIONAL/1e6:.0f}mm")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

**Key takeaway:** at SPX=7200, the SPX hedge size ranges from $20mm (USDJPY=150) to $50mm (USDJPY=162). A 6-yen FX move *doubles* the SPX hedge requirement at the same SPX spot — the cross-gamma, embedded in the static delta surface.

## 6. SPX vega — bell-shaped, fanned by FX level

Same call-vega bell shape as a vanilla, but cross-vega makes the height depend on FX spot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for X0 in fx_levels:
    vegas = [vega_at(s, X0, "sig_S") * units / 1e3 for s in spx_grid]
    ax.plot(spx_grid, vegas, label=f"USDJPY = {X0}", linewidth=2)
vanilla_v = [vega_at(s, 1000.0, "sig_S") * units / 1e3 for s in spx_grid]
ax.plot(spx_grid, vanilla_v, "k--", alpha=0.6, label="Vanilla", linewidth=1.5)
ax.axvline(p.K, color="gray", ls=":", alpha=0.6, label=f"Strike {p.K:.0f}")
ax.axvline(p.S0, color="red", ls=":", alpha=0.5)
ax.set_xlabel("SPX spot"); ax.set_ylabel("SPX vega ($k per vol pt)")
ax.set_title("SPX Vega vs Spot, by USDJPY Level")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Cross-vega is severe:** at the strike, vega ranges from $80k/vol pt (USDJPY=150) to $230k/vol pt (USDJPY=162). The vega book gets dramatically longer in a JPY-weakening regime — exactly when SPX vol is typically bid in risk-on episodes.

## 7. FX vega — flips sign at the barrier

The FX leg is a digital. Below the barrier, you *want* volatility (more chance of breaching). Above, vol is your enemy. At the barrier, vega is exactly zero.

In [ ]:
fx_grid = np.linspace(140, 175, 80)
spx_levels = [6600, 6900, 7200, 7500, 7800]

fig, ax = plt.subplots(figsize=(10, 5.5))
for S0 in spx_levels:
    fxv = [vega_at(S0, x, "sig_X") * units / 1e3 for x in fx_grid]
    ax.plot(fx_grid, fxv, label=f"SPX = {S0}", linewidth=2)

ax.axvline(p.B, color="k", ls="--", alpha=0.6, label=f"Barrier {p.B:.0f}")
ax.axvline(p.X0, color="red", ls=":", alpha=0.5, label=f"USDJPY {p.X0:.0f}")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("USDJPY spot"); ax.set_ylabel("FX vega ($k per vol pt)")
ax.set_title("FX Vega vs USDJPY Spot — sign flips at the barrier")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Hedging implication:** an ATM USDJPY straddle has positive vega everywhere. To hedge a position whose vega flips sign, you'd need a **risk reversal centered at the barrier** — itself a correlation-sensitive instrument.

## 8. Cega — bell-shaped, peaks just below the barrier

Correlation matters most when *both* legs are individually uncertain. As USDJPY moves to either extreme, one leg becomes deterministic and cega collapses.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for S0 in spx_levels:
    cs = [cega_at(S0, x) * units / 1e3 for x in fx_grid]
    ax.plot(fx_grid, cs, label=f"SPX = {S0}", linewidth=2)

ax.axvline(p.B, color="k", ls="--", alpha=0.6, label=f"Barrier {p.B:.0f}")
ax.axvline(p.X0, color="red", ls=":", alpha=0.5)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("USDJPY spot"); ax.set_ylabel("Cega ($k per 1% corr)")
ax.set_title("Cega vs USDJPY Spot — peaks just below barrier")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**At current spot,** cega ≈ $30k per 1% correlation — very near the peak. The trade sits at maximum correlation risk per dollar of premium.

The peak is *below* the barrier (not at it) due to lognormal asymmetry: the median path lands a bit below spot over 6 months.

## 9. Cross-gamma surface — the silent hedge killer

$\partial \Delta_S / \partial X$: how much SPX delta changes per yen of USDJPY move. This is the cross-Greek that makes a "perfectly delta-hedged" position misbehave when FX moves.

In [ ]:
spx_g = np.linspace(p.S0 - 1100, p.S0 + 900, 35)
fx_g = np.linspace(p.X0 - 11, p.X0 + 14, 35)
SS, XX = np.meshgrid(spx_g, fx_g)

dS, dX = 0.005 * p.K, 0.5
cg = np.zeros_like(SS)
for i in range(SS.shape[0]):
    for j in range(SS.shape[1]):
        s, x = SS[i, j], XX[i, j]
        d_up = (price_at(s + dS, x + dX) - price_at(s - dS, x + dX)) / (2 * dS)
        d_dn = (price_at(s + dS, x - dX) - price_at(s - dS, x - dX)) / (2 * dS)
        cg[i, j] = (d_up - d_dn) / (2 * dX)

fig, ax = plt.subplots(figsize=(9, 6))
vmax = np.abs(cg).max()
cf = ax.contourf(SS, XX, cg, levels=20, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.axhline(p.B, color="k", ls="--", alpha=0.5, label="Barrier")
ax.axvline(p.K, color="gray", ls=":", alpha=0.5, label="Strike")
ax.scatter([p.S0], [p.X0], color="white", edgecolor="black", s=100, zorder=5, label="Spot")
plt.colorbar(cf, ax=ax, label=r"$\partial \Delta_S / \partial X$")
ax.set_xlabel("SPX spot"); ax.set_ylabel("USDJPY spot")
ax.set_title("Cross-gamma: SPX delta sensitivity to USDJPY")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()

**The cross-gamma is largest in the SPX-high / FX-just-below-barrier region** — exactly where a "successful" trade is heading. As the position becomes valuable, your SPX hedge becomes more sensitive to FX moves you weren't planning to hedge.

## 10. Scenario: deep ITM, delta-hedged, sharp FX shock

The classic pathology of EQ/FX hybrids. You think you're risk-neutral on SPX. You aren't.

**State 0:** SPX = 7600 (deep ITM), USDJPY = 162 (FX leg ITM), SPX delta-hedged.

**Shock:** USDJPY drops 6 yen to 156 — a single liquid trading day in the pair.

In [ ]:
S0_init, X0_init = 7600.0, 162.0
P0 = price_at(S0_init, X0_init)
D0 = delta_at(S0_init, X0_init)
units0 = NOTIONAL / S0_init
hedge_units = D0 * units0
opt0 = P0 * units0

print(f"State 0:")
print(f"  Option MTM:    ${opt0/1e6:.2f}mm")
print(f"  SPX delta:     {D0:.3f}")
print(f"  Hedge:         short {hedge_units:,.0f} SPX = ${hedge_units*S0_init/1e6:.2f}mm")
print()

scenarios = [
    ("(a) FX -6, SPX flat", S0_init,        156.0),
    ("(b) FX -6, SPX -2%",  S0_init * 0.98, 156.0),
    ("(c) FX -6, SPX -4%",  S0_init * 0.96, 156.0),
]

print(f"{'Scenario':<24}{'OptPnL':>10}{'HedgePnL':>11}{'NetPnL':>10}")
print("-" * 56)
for name, S1, X1 in scenarios:
    opt_pnl = (price_at(S1, X1) - P0) * units0
    hedge_pnl = hedge_units * (S0_init - S1)
    net = opt_pnl + hedge_pnl
    print(f"{name:<24}{opt_pnl/1e6:>+10.3f}{hedge_pnl/1e6:>+11.3f}{net/1e6:>+10.3f}")

On a 6-yen FX move with SPX flat, you lose ~$2mm — about 26% of MTM — despite a "perfect" SPX hedge.

The 2D PnL surface below shows why: the breakeven contour is steeply tilted, because the hedge captures SPX risk but leaves FX exposure naked.

In [ ]:
spx_shocks = np.linspace(-0.06, 0.04, 35)
fx_shocks = np.linspace(-8, 2, 35)
PNL = np.zeros((len(fx_shocks), len(spx_shocks)))
for i, dx in enumerate(fx_shocks):
    for j, ds in enumerate(spx_shocks):
        S1 = S0_init * (1 + ds); X1 = X0_init + dx
        opt_pnl = (price_at(S1, X1) - P0) * units0
        hedge_pnl = hedge_units * (S0_init - S1)
        PNL[i, j] = (opt_pnl + hedge_pnl) / 1e6

fig, ax = plt.subplots(figsize=(9, 6))
vmax = np.abs(PNL).max()
cf = ax.contourf(spx_shocks * 100, fx_shocks, PNL,
                 levels=np.linspace(-vmax, vmax, 21), cmap="RdYlGn")
ax.contour(spx_shocks * 100, fx_shocks, PNL, levels=[0], colors="black", linewidths=1.2)
plt.colorbar(cf, ax=ax, label="Net PnL ($mm)")
ax.scatter([0], [0], color="white", edgecolor="black", s=80, zorder=5, label="Start")
ax.scatter([0], [-6], color="black", s=80, marker="X", zorder=5, label="(a) FX-6, SPX flat")
ax.scatter([-2], [-6], color="black", s=80, marker="^", zorder=5, label="(b) FX-6, SPX-2%")
ax.set_xlabel("SPX shock (%)"); ax.set_ylabel("USDJPY shock (yen)")
ax.set_title("Net PnL: delta-hedged at start, hedge held static")
ax.legend(loc="upper right", fontsize=9)
ax.axhline(0, color="black", linewidth=0.4); ax.axvline(0, color="black", linewidth=0.4)
plt.tight_layout(); plt.show()

## Takeaways

1. **SPX delta hedging gives false comfort.** The trade has three independent risks (SPX, FX, correlation); hedging one doesn't help when the others move.
2. **FX vega flips sign at the barrier.** Standard FX vol products can't hedge this — you need a barrier-centered risk reversal.
3. **Cega peaks just below the barrier and decays away.** Correlation risk is maximal while the trade is "live."
4. **Cross-Greeks dominate.** Cross-gamma, cross-vega, and FX-conditional vanna are where the real risk lives — and none appear in vanilla option books.

For a real desk mark, layer on SPX skew, FX skew, correlation skew, and stochastic vol. This bivariate BS is the benchmark, not the production mark.